# TT-SVD: 3階テンソルでの基礎実験（src版）

元Notebook [`00_tt_svd_3way_basics.ipynb`](../00_fundamentals/00_tt_svd_3way_basics.ipynb) では、3階テンソルに対して **2回のSVDを手で実装** してTT coreのshapeを追いました。

このsrc版では、同じ現象を `nn_compression.compression` のTT APIで再確認します。

## ゴール

- $X \in \mathbb{R}^{n_1 \times n_2 \times n_3}$ を `tt_svd_exact` で3個のTT coreへ分解する
- core shapeと bond rank を確認する
- 打ち切りなしなら元テンソルを数値誤差の範囲で再構成できることを確認する
- 第1cut unfolding rankと bond rank の対応を確認する

## 手実装版との対応

手実装版では次のcore shapeを確認しました。

- $G_1 \in \mathbb{R}^{1 \times n_1 \times r_1}$
- $G_2 \in \mathbb{R}^{r_1 \times n_2 \times r_2}$
- $G_3 \in \mathbb{R}^{r_2 \times n_3 \times 1}$

> gauge freedom 以降は扱わない。

## 1. 実験対象テンソル

$X.shape = (2, 3, 4)$ の小さいテンソルを使う。元Notebookと同じ `float64` と seed `0` を使う。

In [1]:
import torch

from nn_compression.compression import (
    tt_svd_exact,
    tt_unfold,
    tt_reconstruct,
)
from nn_compression.metrics import relative_frobenius_error

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

X = torch.randn(2, 3, 4)
n1, n2, n3 = X.shape

print(f"X.shape = {tuple(X.shape)}")
print(f"X.ndim = {X.ndim}")
print(f"X.numel() = {X.numel()}")

X.shape = (2, 3, 4)
X.ndim = 3
X.numel() = 24


## 2. 第1切断のunfolding

最初の切断は $i_1 \mid i_2 i_3$ なので、行列shapeは $(n_1,\ n_2 n_3)$ になる。

TT cut unfolding には `tt_unfold(X, 1)` を使う。

In [2]:
X1 = tt_unfold(X, 1)
print(f"X1.shape = {tuple(X1.shape)}")  # (2, 12)

r1 = int(torch.linalg.matrix_rank(X1).item())
print(f"r1 = {r1}")

X1.shape = (2, 12)
r1 = 2


## 3. exact TT-SVD

手実装版では2回SVDして $G_1, G_2, G_3$ を作りました。src版では `tt_svd_exact(X)` を1回呼び出す。

各coreは

$$
G^{(k)} \in \mathbb{R}^{r_{k-1} \times n_k \times r_k}
$$

のshapeを持つ。

In [3]:
cores = tt_svd_exact(X)

for i, core in enumerate(cores, start=1):
    print(f"G{i}.shape = {tuple(core.shape)}")

bond_ranks = [core.shape[2] for core in cores[:-1]]
print(f"bond_ranks = {bond_ranks}")

G1.shape = (1, 2, 2)
G2.shape = (2, 3, 4)
G3.shape = (4, 4, 1)
bond_ranks = [2, 4]


## 4. TT coreから再構成

3個のcoreを `tt_reconstruct` で縮約し、relative Frobenius errorを確認する。

In [4]:
X_hat = tt_reconstruct(cores)
rel_error = relative_frobenius_error(X, X_hat)

print(f"X_hat.shape = {tuple(X_hat.shape)}")
print(f"relative Frobenius error = {rel_error.item():.3e}")

X_hat.shape = (2, 3, 4)
relative Frobenius error = 5.792e-15


## 5. 確認

- `G1.shape == (1, n1, r1)`
- `G2.shape == (r1, n2, r2)`
- `G3.shape == (r2, n3, 1)`
- 打ち切りなしなら相対誤差は数値誤差の範囲
- bond rankは各cut unfolding rankと一致する

次は一般 $d$ 階へ拡張し、TT-rankと元テンソルのunfolding rankを直接照合する（`01_tt_rank_unfolding.ipynb` src版）。